# Forward Factor Scanner, step by step

> **Disclaimer.** Educational and research use only. This is not investment advice.

## What this scanner is for

An option's implied volatility (IV) is an average over its whole life. Two expiries on the same stock tell you more than either one alone.
From a 30-day IV and a 60-day IV you can back out the volatility the market expects **between** day 30 and day 60. That is the **forward volatility**.

The **forward factor** (FF) compares the front expiry's IV with that forward vol:

$$FF = \frac{\sigma_1 - \sigma_{fwd}}{\sigma_{fwd}}$$

A high FF means the front month is expensive compared with the period after it.
A **calendar spread** sells the front expiry and buys the back expiry at the same strike, so it profits if that gap closes. A high FF is the setup it wants.

The scanner takes a ticker, picks front expiries near 30 DTE and back expiries near 60 DTE, and scores every front/back pair as three kinds of trade:

| Structure | Legs |
|---|---|
| `call` | call calendar at the strike nearest spot and one strike step either side. The nearest one is the **ATM** call calendar, the default trade |
| `put` | put calendar at the same strikes |
| `double_35d` | call calendar at the +35 delta strike plus put calendar at the −35 delta strike |

A trade **passes** when all four rules hold:

| Rule | Flag when it fails | Pass when |
|---|---|---|
| Forward factor | `ff` | FF ≥ 0.20 |
| Earnings | `earnings` | no earnings date between today and the back expiry |
| Volume | `volume` | every leg traded ≥ 100 contracts today |
| Open interest | `oi` | every leg has ≥ 1,000 contracts open |

Trades are ranked by FF, highest first, with trades blocked by earnings or liquidity pushed below the rest.

## How this notebook is organised

Section 2 works through the forward vol maths on hand-typed numbers, the way the original GUI calculator did.
Sections 3 to 10 then fetch real data and build the scan **one small function** at a time, printing every intermediate value.
Section 11 joins the pieces into `scan()`, the single function the rest of the code calls.

That is the same split as chrysos-lab: `src/calculators/forward_factor.py` holds the maths for one pair,
and `src/scanners/forward_factor.py` chooses which pairs and strikes to feed it.

The notebook is self-contained. It doesn't import anything from the repo, so it runs on Google Colab as-is.

## 0. Setup

Colab already has `numpy`, `pandas` and `matplotlib`. Only `yfinance` (free Yahoo Finance data) needs installing.

In [ ]:
%pip install -q yfinance

In [ ]:
import logging
import math
from dataclasses import dataclass, field, replace
from datetime import date, datetime, timedelta
from typing import Literal
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

pd.set_option("display.width", 120)

MARKET_TZ = ZoneInfo("America/New_York")


class CalculatorError(ValueError):
    """Raised when the calculator can't produce a result from the data it was given."""


def market_date(now=None):
    """Today's date on US exchanges, whatever the local time zone."""
    return (now or datetime.now(MARKET_TZ)).astimezone(MARKET_TZ).date()

### Choose a ticker and a data source

- `SYMBOL`: the stock to scan.
- `DATA_SOURCE`:
  - `"yahoo"` uses live Yahoo Finance data.
  - `"synthetic"` generates realistic fake data, so the notebook runs at any hour.
- `AS_OF`: the date DTE is counted from. It is today's date **in New York**, not your local date.
  East of New York, local midnight arrives hours earlier, and counting from the local date would make every DTE a day short for part of the day.

**Timing matters for `"yahoo"`.** US stock options trade only from 9:30 AM to 4:00 PM New York time.
After the close, Yahoo keeps serving the last quotes for a while. Overnight and at weekends it clears them: bid/ask become 0 and IV drops to a placeholder of 0.00001.
If you run during that window, Step 7 stops with a "stale quotes" error. Switch to `"synthetic"` or try again after the US open.

In [ ]:
SYMBOL = "AAPL"
DATA_SOURCE = "yahoo"  # "yahoo" or "synthetic"
AS_OF = market_date()  # the US date that days-to-expiry (DTE) is counted from
AS_OF

## 1. Parameters

All thresholds live in two frozen dataclasses, matching the two modules in the repo:

- `FFParams` is the maths for one pair (`calculators.forward_factor.Params`): the FF pass mark and the day count that turns DTE into years.
- `ScanParams` is the scanner (`scanners.forward_factor.Params`): which expiries and strikes to try, and the liquidity floors.

| Field | Meaning | CLI flag |
|---|---|---|
| `front_window`, `back_window` | DTE ranges around 30 and 60, both ends included | |
| `widen_step`, `max_widen_steps` | an empty window grows by 7 days at each end, up to 6 times | |
| `strike` | `None` scans strikes around spot; a number scans that strike only | `--strike` |
| `strike_range` | strike steps either side of the strike nearest spot | `--range` |
| `min_volume`, `min_open_interest` | per-leg liquidity floors | `--min-volume`, `--min-oi` |
| `double_delta`, `rate` | wing delta of the double calendar, and the risk-free rate used for deltas | |

In [ ]:
@dataclass(frozen=True)
class FFParams:
    min_ff: float = 0.2
    days_per_year: float = 365.0


@dataclass(frozen=True)
class ScanParams:
    front_window: tuple = (21, 45)  # around 30 DTE
    back_window: tuple = (50, 90)  # around 60 DTE
    widen_step: int = 7  # days added to each end of an empty window
    max_widen_steps: int = 6
    strike: float | None = None  # scan only this strike; None means a window around spot
    strike_range: int = 1  # strike steps above and below the strike nearest spot
    min_volume: float = 100  # contracts each leg must have traded today
    min_open_interest: float = 1_000  # open contracts each leg must have
    double_delta: float = 0.35  # wing delta of the double calendar
    rate: float = 0.0  # risk-free rate used for deltas
    calculator: FFParams = FFParams()


params = ScanParams()
params

## 2. `evaluate_pair`: forward vol and FF for one pair

IV is quoted annualised, so first turn each leg into **total variance**, the variance accumulated up to its expiry:
$\sigma^2 T$ with $T = \text{DTE} / 365$. Variances add up over time (vols don't), so the variance between the two expiries is the difference:

$$\sigma_{fwd}^2 = \frac{\sigma_2^2 T_2 - \sigma_1^2 T_1}{T_2 - T_1} \qquad FF = \frac{\sigma_1 - \sigma_{fwd}}{\sigma_{fwd}}$$

FF = 0.20 means the front IV is 20% above the forward vol. That is the pass mark.

If the back leg's total variance is **smaller** than the front's, forward variance comes out negative. No real forward vol fits those two quotes, which usually means a bad quote.
Forward vol and FF are then `None` and the trade fails.

Each check is stored as a `Check(value, threshold, op, passed)`, so the result keeps the **number**, not just pass/fail.
The result keeps every intermediate value too, so the working can be printed.

In [ ]:
@dataclass(frozen=True)
class Check:
    value: float
    threshold: float
    op: Literal[">=", "<="]
    passed: bool


def at_least(value, threshold):
    return Check(value, threshold, ">=", value >= threshold)


def total_variance(iv, t):
    return iv**2 * t


def forward_variance(iv1, t1, iv2, t2):
    return (total_variance(iv2, t2) - total_variance(iv1, t1)) / (t2 - t1)


@dataclass(frozen=True)
class PairInputs:
    """Front (1) and back (2) legs. IVs are annualised decimals (0.245 = 24.5%)."""

    dte1: float
    iv1: float
    dte2: float
    iv2: float


@dataclass(frozen=True)
class FFResult:
    t1: float  # years
    t2: float
    iv1: float
    iv2: float
    tv1: float  # total variance iv1^2 * t1
    tv2: float
    fwd_var: float
    fwd_vol: float | None  # None when forward variance is negative
    ff: Check | None  # None when fwd_vol is None or zero


def evaluate_pair(inputs, params=FFParams()):
    if inputs.dte1 < 0 or inputs.dte2 < 0:
        raise CalculatorError("DTEs must be non-negative.")
    if inputs.dte2 <= inputs.dte1:
        raise CalculatorError("Require DTE2 > DTE1 so that T2 > T1.")
    if inputs.iv1 < 0 or inputs.iv2 < 0:
        raise CalculatorError("IVs must be non-negative.")

    t1 = inputs.dte1 / params.days_per_year
    t2 = inputs.dte2 / params.days_per_year
    fwd_var = forward_variance(inputs.iv1, t1, inputs.iv2, t2)
    fwd_vol = math.sqrt(fwd_var) if fwd_var >= 0 else None
    ff = at_least((inputs.iv1 - fwd_vol) / fwd_vol, params.min_ff) if fwd_vol else None

    return FFResult(
        t1=t1, t2=t2, iv1=inputs.iv1, iv2=inputs.iv2,
        tv1=total_variance(inputs.iv1, t1), tv2=total_variance(inputs.iv2, t2),
        fwd_var=fwd_var, fwd_vol=fwd_vol, ff=ff,
    )

The working is printed as a table, the same one the terminal command prints under each ranked trade.

In [ ]:
CHECK_HEADERS = ("Item", "Result", "Value", "Threshold")


def format_table(headers, rows):
    """Left-aligned columns: header line, dashed rule, then one line per row."""
    widths = [max(len(row[i]) for row in (headers, *rows)) for i in range(len(headers))]
    join = lambda cells: "  ".join(c.ljust(w) for c, w in zip(cells, widths)).rstrip()
    return [join(headers), join(["-" * w for w in widths]), *(join(row) for row in rows)]


def check_row(name, check, fmt):
    return (name, "PASS" if check.passed else "FAIL", fmt.format(check.value), f"{check.op} {fmt.format(check.threshold)}")


def breakdown_rows(r):
    fwd_var = f"{r.fwd_var:.8f}"
    if r.fwd_vol is None:
        fwd_var += " (negative)"
        fwd_vol, ff_note = "n/a (check inputs)", "negative fwd variance"
    else:
        fwd_vol, ff_note = f"{r.fwd_vol:.4%}", "fwd vol is 0"

    rows = [
        ("T1 = DTE1 / 365", "", f"{r.t1:.6f} yr", ""),
        ("sigma1", "", f"{r.iv1:.6f}", ""),
        ("total variance1", "", f"{r.tv1:.8f}", ""),
        ("T2 = DTE2 / 365", "", f"{r.t2:.6f} yr", ""),
        ("sigma2", "", f"{r.iv2:.6f}", ""),
        ("total variance2", "", f"{r.tv2:.8f}", ""),
        ("forward variance", "", fwd_var, ""),
        ("forward vol", "", fwd_vol, ""),
    ]
    if r.ff is None:
        rows.append(("ff", "FAIL", f"undefined ({ff_note})", ""))
    else:
        rows += [check_row("ff", r.ff, "{:.4f}"), ("ff (%)", "", f"{r.ff.value:.2%}", "")]
    return rows


def format_breakdown(r):
    return "\n".join(format_table(CHECK_HEADERS, breakdown_rows(r)))

### A worked example

The original GUI calculator took four numbers by hand. Try a 30-day option at 32% IV and a 60-day option at 29%.

Only 3 vol points separate the two legs, yet FF comes out around 0.25 and passes.
The 60-day IV already includes the first 30 days at 32%, so to average 29% overall the second 30 days must be much quieter: about 25.7%.

In [ ]:
example = evaluate_pair(PairInputs(dte1=30, iv1=0.32, dte2=60, iv2=0.29), params.calculator)
print(format_breakdown(example))

Picture it as total variance against time. Each expiry is a point $(T, \sigma^2 T)$, and the **slope** of a line is a variance per year.

- From the origin to the front point, the slope is $\sigma_1^2$.
- Between the two points, the slope is the forward variance.

A rich front month lifts the front point, which flattens the segment after it. The flatter that segment, the lower the forward vol and the higher FF.

In [ ]:
e = example
plt.figure(figsize=(8, 4))
plt.plot([0, e.t2], [0, e.tv2], color="#52514e", ls=":", lw=2, label=f"back average: slope IV2² = {e.iv2**2:.4f}")
plt.plot([0, e.t1], [0, e.tv1], color="#2a78d6", lw=2, label=f"front: slope IV1² = {e.iv1**2:.4f}")
plt.plot([e.t1, e.t2], [e.tv1, e.tv2], color="#eb6834", lw=2, label=f"forward: slope = fwd var = {e.fwd_var:.4f}")
plt.scatter([e.t1, e.t2], [e.tv1, e.tv2], s=64, color="#0b0b0b", zorder=3)
for t, tv, name in ((e.t1, e.tv1, "front"), (e.t2, e.tv2, "back")):
    plt.annotate(f"{name} σ²T = {tv:.5f}", (t, tv), textcoords="offset points", xytext=(-10, 6), ha="right")
plt.margins(y=0.12)
plt.xlabel("T (years)"); plt.ylabel("total variance σ²T"); plt.title("Forward variance is the slope between two expiries"); plt.legend()
plt.show()

And the failure case: a 30-day option at 50% next to a 35-day option at 40%.
The back leg would have to carry less total variance than the front, which is impossible, so forward variance is negative and FF is undefined.

In [ ]:
print(format_breakdown(evaluate_pair(PairInputs(dte1=30, iv1=0.50, dte2=35, iv2=0.40), params.calculator)))

## 3. Data providers

The scanner never talks to Yahoo directly. It asks a **provider** for four things:

| Method | Returns |
|---|---|
| `expirations(symbol)` | all listed option expiry dates |
| `option_chain(symbol, expiry)` | calls and puts for one expiry, with columns `strike, iv, bid, ask, volume, open_interest` |
| `spot(symbol)` | the latest stock price |
| `next_earnings(symbol, after)` | the next earnings date on or after `after`, or `None` |

Every provider returns the **same column names**, whatever the source calls them.
Yahoo uses `impliedVolatility` and `openInterest`, so the Yahoo provider renames them to `iv` and `open_interest`.
`volume` is contracts traded today (NaN when none). Open interest is as of the last close.

Unlike the calendar IV crush calculator, there's no price history here: FF uses only implied vols.

In [ ]:
@dataclass(frozen=True)
class OptionChain:
    expiry: date
    calls: pd.DataFrame  # columns: strike, iv, bid, ask, volume, open_interest
    puts: pd.DataFrame

    def side(self, side):
        return self.calls if side == "call" else self.puts


SIDES = ("call", "put")
CHAIN_COLUMNS = ["strike", "iv", "bid", "ask", "volume", "open_interest"]


class YFinanceProvider:
    def __init__(self):
        self._tickers = {}

    def _ticker(self, symbol):
        if symbol not in self._tickers:
            self._tickers[symbol] = yf.Ticker(symbol)
        return self._tickers[symbol]

    def expirations(self, symbol):
        try:
            raw = self._ticker(symbol).options
        except Exception:
            return []
        return [datetime.strptime(d, "%Y-%m-%d").date() for d in raw]

    def option_chain(self, symbol, expiry):
        chain = self._ticker(symbol).option_chain(expiry.strftime("%Y-%m-%d"))
        rename = {"impliedVolatility": "iv", "openInterest": "open_interest"}
        normalise = lambda df: df.rename(columns=rename)[CHAIN_COLUMNS]
        return OptionChain(expiry, normalise(chain.calls), normalise(chain.puts))

    def spot(self, symbol):
        today = self._ticker(symbol).history(period="1d")
        return None if today.empty else float(today["Close"].iloc[-1])

    def next_earnings(self, symbol, after):
        # Yahoo has no calendar for ETFs and indices; yfinance logs that 404 as an error.
        logger = logging.getLogger("yfinance")
        level = logger.level
        logger.setLevel(logging.CRITICAL)
        try:
            dates = self._ticker(symbol).calendar.get("Earnings Date") or []
        except Exception:
            return None
        finally:
            logger.setLevel(level)
        days = [d.date() if isinstance(d, datetime) else d for d in dates if isinstance(d, date)]
        return min((d for d in days if d >= after), default=None)

The synthetic provider builds a stock at 100 with weekly Friday expiries:

- front-month IV is rich and decays with DTE towards about 28%;
- an earnings date about 66 days out adds a 7% move's worth of variance to every expiry after it;
- the skew lifts IV on lower strikes;
- monthly expiries (the third Friday) trade far more than weeklies, so some weekly legs fail the volume or open interest floors.

It is only there so you can follow every step when Yahoo has no live quotes.

In [ ]:
class SyntheticProvider:
    def __init__(self, as_of, spot=100.0, earnings_in=66, earnings_move=0.07):
        self.as_of, self._spot = as_of, spot
        self._earnings = as_of + timedelta(days=earnings_in)
        self._event_var = earnings_move**2

    def expirations(self, symbol):
        first_friday = self.as_of + timedelta(days=(4 - self.as_of.weekday()) % 7 or 7)
        return [first_friday + timedelta(weeks=w) for w in range(20)]

    def option_chain(self, symbol, expiry):
        dte = (expiry - self.as_of).days
        t = dte / 365
        base_iv = 0.28 + 0.20 * math.exp(-dte / 20)  # rich front month, settling to ~0.28
        event_var = self._event_var if expiry >= self._earnings else 0.0
        atm_iv = math.sqrt(base_iv**2 + event_var / t)  # the earnings move adds its variance once
        strikes = np.arange(self._spot * 0.7, self._spot * 1.3 + 0.01, 2.5)
        moneyness = np.log(strikes / self._spot)
        iv = atm_iv - 0.25 * moneyness + 0.8 * moneyness**2  # put skew plus a smile

        monthly = 15 <= expiry.day <= 21  # third Friday
        volume = (4_000 * (1.0 if monthly else 0.15) * math.exp(-dte / 60) * np.exp(-np.abs(moneyness) / 0.04)).round()
        open_interest = (30_000 * (1.0 if monthly else 0.05) * np.exp(-np.abs(moneyness) / 0.05)).round()
        time_value = 0.4 * self._spot * iv * math.sqrt(t) * np.exp(-np.abs(moneyness) / 0.1)

        def side(intrinsic, iv_shift):
            mid = intrinsic + time_value
            return pd.DataFrame({"strike": strikes, "iv": iv + iv_shift, "bid": mid * 0.97, "ask": mid * 1.03,
                                 "volume": volume, "open_interest": open_interest})

        return OptionChain(expiry, side(np.maximum(self._spot - strikes, 0), 0.0), side(np.maximum(strikes - self._spot, 0), 0.003))

    def spot(self, symbol):
        return self._spot

    def next_earnings(self, symbol, after):
        return self._earnings if self._earnings >= after else None


provider = YFinanceProvider() if DATA_SOURCE == "yahoo" else SyntheticProvider(AS_OF)
symbol = SYMBOL.strip().upper()

all_expiries = provider.expirations(symbol)
if not all_expiries:
    raise CalculatorError(f"No options found for stock symbol '{symbol}'.")
print(f"{symbol}: {len(all_expiries)} listed expiries")
print([e.isoformat() for e in all_expiries[:12]], "..." if len(all_expiries) > 12 else "")

## 4. `pick_legs`: which expiries to pair

Rather than one front and one back expiry, the scanner keeps **every** listed expiry in each window:
front 21-45 DTE, back 50-90 DTE. Three fronts and four backs make twelve pairs to compare.

1. Expiries on `as_of` are dropped.
2. A back expiry must come after the earliest front expiry.
3. If a window holds no listed expiry at all, as with names that only list monthlies, it widens by 7 days at each end and tries again, up to 6 times.

A window is **never** widened just because no trade passes. That would quietly change what is being screened.

In [ ]:
@dataclass(frozen=True)
class Legs:
    front: list
    back: list
    front_window: tuple
    back_window: tuple
    widened: bool


def window_label(window):
    return f"{window[0]}-{window[1]}"


def widen(window, step):
    return (max(1, window[0] - step), window[1] + step)


def pick_legs(expiries, as_of, params=ScanParams()):
    dtes = {expiry: (expiry - as_of).days for expiry in sorted(set(expiries)) if expiry > as_of}
    front_window, back_window = params.front_window, params.back_window

    for step in range(params.max_widen_steps + 1):
        front = [e for e, d in dtes.items() if front_window[0] <= d <= front_window[1]]
        back = [e for e, d in dtes.items() if back_window[0] <= d <= back_window[1] and (not front or e > front[0])]
        if front and back:
            return Legs(front, back, front_window, back_window, widened=step > 0)
        if step == params.max_widen_steps:
            break
        if not front:
            front_window = widen(front_window, params.widen_step)
        if not back:
            back_window = widen(back_window, params.widen_step)

    raise CalculatorError(
        f"No expiry pair fits front {window_label(front_window)} / back {window_label(back_window)} DTE, even after widening."
    )


legs = pick_legs(all_expiries, AS_OF, params)
dte_of = lambda expiry: (expiry - AS_OF).days

print(f"Front window {window_label(legs.front_window)} DTE, back window {window_label(legs.back_window)} DTE"
      + ("  (widened)" if legs.widened else ""))
shown = [e for e in sorted(all_expiries) if dte_of(e) <= legs.back_window[1] + 14]
pd.DataFrame({"expiry": shown, "dte": [dte_of(e) for e in shown],
              "leg": ["front" if e in legs.front else "back" if e in legs.back else "" for e in shown]})

## 5. Fetch the chains, spot price and earnings date

Only the expiries in `legs` are downloaded. Fetching chains is the slow, rate-limited part.

Below is the strip of the first front chain's calls around spot, already normalised.
The row whose strike is nearest a target is what `nearest_quote` returns; with spot as the target, that is the at-the-money (ATM) quote.

In [ ]:
def nearest_quote(side, target):
    """The row of a calls or puts frame whose strike is closest to ``target``."""
    return side.loc[(side["strike"] - target).abs().idxmin()]


chains = {e: provider.option_chain(symbol, e) for e in sorted({*legs.front, *legs.back})}

spot = provider.spot(symbol)
if spot is None:
    raise CalculatorError("Unable to retrieve underlying stock price.")
earnings = provider.next_earnings(symbol, AS_OF)

print(f"Spot price:    {spot:,.2f}")
print(f"Next earnings: {earnings or 'none found'}")
front_calls = chains[legs.front[0]].calls
front_calls.iloc[(front_calls["strike"] - spot).abs().argsort()[:7]].sort_values("strike")

The ATM call IV of each fetched expiry, with both windows shaded. FF is high where the curve drops steeply from a front expiry to a back expiry.
An earnings date shows up as a step: every expiry after it carries the earnings move, which lifts back-month IV and pulls FF down.

In [ ]:
atm_ivs = [float(nearest_quote(c.calls, spot)["iv"]) for c in chains.values()]

plt.figure(figsize=(8, 4))
for (lo, hi), color, name in ((legs.front_window, "#2a78d6", "front"), (legs.back_window, "#eb6834", "back")):
    plt.axvspan(lo, hi, color=color, alpha=0.15, label=f"{name} window {lo}-{hi} DTE")
plt.plot([dte_of(e) for e in chains], atm_ivs, color="#0b0b0b", lw=2, marker="o", ms=8, label="ATM call IV")
if earnings and dte_of(earnings) <= legs.back_window[1] + 14:
    plt.axvline(dte_of(earnings), color="#52514e", ls="--", lw=2, label=f"earnings {earnings}")
plt.xlabel("Days to expiry"); plt.ylabel("ATM implied vol"); plt.title(f"{symbol} IV term structure ({DATA_SOURCE} data)"); plt.legend()
plt.show()

## 6. `strikes_around`: which strikes to try

From here on we follow **one pair** through every step: the back expiry nearest 60 DTE, and the front expiry nearest 30 DTE before it.

A calendar needs the same strike listed on **both** expiries. `listed_strikes` keeps the strikes every chain lists, and each side gets its own ladder,
because a strike can be listed for calls but not puts. `strikes_around` sorts that shared ladder, finds the strike nearest spot,
and takes `strike_range` rungs either side. Counting rungs rather than dollars works whatever the increment: $1 strikes on a $20 stock, $5 on a $500 one.
Ties go to the lower strike.

The **ATM** call calendar is `strikes_around` with a width of 0 on the call ladder.

In [ ]:
def listed_strikes(chains, sides=SIDES):
    """Strikes listed on every given side of every chain."""
    strikes = None
    for chain in chains:
        listed = set.intersection(*(set(chain.side(side)["strike"]) for side in sides))
        strikes = listed if strikes is None else strikes & listed
    return strikes or set()


def strikes_around(chains, target, width, sides=SIDES):
    """The listed strike nearest ``target`` plus up to ``width`` listed strikes either side, ascending."""
    ladder = sorted(listed_strikes(chains, sides))
    if not ladder:
        return []
    center = min(range(len(ladder)), key=lambda i: (abs(ladder[i] - target), ladder[i]))
    return ladder[max(0, center - width) : center + width + 1]


demo_back = min(legs.back, key=lambda e: abs(dte_of(e) - 60))
demo_front = min((e for e in legs.front if e < demo_back), key=lambda e: abs(dte_of(e) - 30))
front, back = chains[demo_front], chains[demo_back]
print(f"Demo pair: front {demo_front} ({dte_of(demo_front)}d) / back {demo_back} ({dte_of(demo_back)}d)")
print(f"Call strikes listed: front {len(front.calls)}, back {len(back.calls)}, both {len(listed_strikes((front, back), ('call',)))}")

call_window = strikes_around((front, back), spot, params.strike_range, sides=("call",))
put_window = strikes_around((front, back), spot, params.strike_range, sides=("put",))
atm_strike = strikes_around((front, back), spot, 0, sides=("call",))[0]
print(f"Spot {spot:,.2f}  |  call strikes {call_window}  |  put strikes {put_window}  |  ATM call strike {atm_strike}")

## 7. `is_stale`: reject cleared quotes

Yahoo clears option quotes overnight and at weekends:

- **bid = 0 and ask = 0**: no market maker is quoting.
- **IV ≤ 0.0001**: Yahoo reports IV as 0.00001 when it can't compute one.

Fed into the maths, those placeholder IVs give nonsense FFs. In the full scan, a trade with any stale leg is **skipped and counted**,
and only when every trade is stale does the scan stop. Here we check the demo pair's ATM legs and stop early with an explanation.

In [ ]:
STALE_IV_FLOOR = 1e-4


def is_stale(quote):
    return (quote["bid"] == 0 and quote["ask"] == 0) or float(quote["iv"]) <= STALE_IV_FLOOR


def stale_error(where):
    return CalculatorError(
        f"Stale option quotes for {where}: bid/ask are zero or IV is at the provider's floor. "
        "Yahoo clears option quotes overnight; try again after the market opens, or set DATA_SOURCE = 'synthetic'."
    )


near = nearest_quote(front.calls, atm_strike)
far = nearest_quote(back.calls, atm_strike)
if is_stale(near) or is_stale(far):
    raise stale_error(f"the {atm_strike:,.2f} calls on {demo_front} / {demo_back}")
print("The demo pair's ATM quotes look live.")
pd.DataFrame({"front": near, "back": far}).T

## 8. FF of the ATM call calendar

Each leg's IV goes into the Section 2 maths: IV1 from the front call, IV2 from the back call, with DTEs counted from `AS_OF`.

`mean_iv` averages the IVs of a leg's quotes. With one quote per leg that's just its IV. It matters for the double calendar in Section 10, which has two quotes per leg.

In [ ]:
def mean_iv(*quotes):
    return sum(float(q["iv"]) for q in quotes) / len(quotes)


pair = PairInputs(dte1=dte_of(demo_front), iv1=mean_iv(near), dte2=dte_of(demo_back), iv2=mean_iv(far))
atm_result = evaluate_pair(pair, params.calculator)
print(f"ATM call calendar @ {atm_strike:,.2f}: {demo_front} / {demo_back}")
print(format_breakdown(atm_result))

## 9. Liquidity, earnings and the close-by date

A good FF isn't a trade on its own. Three more rules decide whether it can be traded. The liquidity rules look at **every** leg and judge the thinnest:

- **Volume**: contracts traded today. A leg nobody traded today may have a stale or wide quote. A missing count (Yahoo's NaN) counts as 0.
- **Open interest**: contracts open as of the last close. Low open interest means wide markets and a hard exit.
- **Earnings**: an earnings date between today and the back expiry fails the trade.
  An event inside the window inflates one leg's IV for a reason that has nothing to do with forward vol being mispriced, so FF stops meaning what it should.

Each trade also gets a **close-by** date: the weekday before the front expiry, so the short leg is closed before it expires. Exchange holidays aren't accounted for.

In [ ]:
def contracts(quote, column):
    """A contract count (``volume`` or ``open_interest``); missing counts as 0."""
    value = quote[column]
    return 0.0 if pd.isna(value) else float(value)


def previous_trading_day(day):
    """The weekday before ``day``. Exchange holidays are not accounted for."""
    day -= timedelta(days=1)
    while day.weekday() >= 5:  # Saturday, Sunday
        day -= timedelta(days=1)
    return day


demo_quotes = (near, far)
volume_check = at_least(min(contracts(q, "volume") for q in demo_quotes), params.min_volume)
oi_check = at_least(min(contracts(q, "open_interest") for q in demo_quotes), params.min_open_interest)
crosses_earnings = earnings is not None and AS_OF <= earnings <= demo_back

print("\n".join(format_table(CHECK_HEADERS, [
    check_row("min leg volume", volume_check, "{:,.0f}"),
    check_row("min leg open int", oi_check, "{:,.0f}"),
    ("earnings", "FAIL" if crosses_earnings else "PASS", str(earnings or "none found"), f"> {demo_back}"),
    ("close by", "", str(previous_trading_day(demo_front)), ""),
])))

Each failed rule becomes a **flag**: `ff`, `earnings`, `volume` or `oi`. A trade with no flags passes.

`make_candidate` does Sections 8 and 9 in one go for any set of legs and returns a `Candidate`. It is checked against the step-by-step values above.

In [ ]:
STRUCTURES = ("call", "put", "double_35d")


@dataclass(frozen=True)
class Candidate:
    structure: str  # "call", "put" or "double_35d"
    front_expiry: date
    front_dte: int
    back_expiry: date
    back_dte: int
    strikes: tuple  # one strike; (put, call) for double_35d
    atm: bool  # the ATM call calendar of its pair
    close_by: date  # weekday before the front expiry
    volume: Check  # thinnest leg's contracts traded today
    open_interest: Check  # thinnest leg's open interest
    crosses_earnings: bool  # earnings between as_of and the back expiry
    result: FFResult
    flags: tuple  # failed rules: "ff", "earnings", "volume", "oi"
    passed: bool


def make_candidate(structure, strikes, atm, nears, fars, front, back, as_of, earnings, params=ScanParams()):
    """Score one trade from its front (near) and back (far) leg quotes."""
    front_dte, back_dte = (front.expiry - as_of).days, (back.expiry - as_of).days
    pair = PairInputs(dte1=front_dte, iv1=mean_iv(*nears), dte2=back_dte, iv2=mean_iv(*fars))
    result = evaluate_pair(pair, params.calculator)

    quotes = (*nears, *fars)
    volume = at_least(min(contracts(q, "volume") for q in quotes), params.min_volume)
    open_interest = at_least(min(contracts(q, "open_interest") for q in quotes), params.min_open_interest)
    crosses_earnings = earnings is not None and as_of <= earnings <= back.expiry
    failed = {
        "ff": result.ff is None or not result.ff.passed,
        "earnings": crosses_earnings,
        "volume": not volume.passed,
        "oi": not open_interest.passed,
    }
    flags = tuple(rule for rule, fail in failed.items() if fail)

    return Candidate(
        structure=structure, front_expiry=front.expiry, front_dte=front_dte, back_expiry=back.expiry, back_dte=back_dte,
        strikes=strikes, atm=atm, close_by=previous_trading_day(front.expiry),
        volume=volume, open_interest=open_interest, crosses_earnings=crosses_earnings,
        result=result, flags=flags, passed=not flags,
    )


atm_candidate = make_candidate("call", (atm_strike,), True, (near,), (far,), front, back, AS_OF, earnings, params)
assert atm_candidate.result == atm_result
assert (atm_candidate.volume, atm_candidate.open_interest) == (volume_check, oi_check)
print("flags:", ", ".join(atm_candidate.flags) or "none", "->", "PASS" if atm_candidate.passed else "FAIL")

## 10. The 35-delta double calendar

A double calendar is two calendars at once: a call calendar above spot and a put calendar below it.
The wings are picked by **delta** rather than by dollar distance, so they sit equally far out in probability terms whatever the stock's vol.
Deltas come from the **front** expiry, the leg being sold, and the back legs use the same strikes.

Black-Scholes delta, no dividends:

$$d_1 = \frac{\ln(S/K) + (r + \sigma^2/2)\,T}{\sigma\sqrt{T}} \qquad \Delta_{call} = N(d_1) \qquad \Delta_{put} = N(d_1) - 1$$

`quote_at_delta` computes each strike's delta with that strike's own IV and keeps the strike closest to the target: +0.35 for the call, −0.35 for the put.
Only strikes listed on both expiries are considered.

The pair is scored on the **averaged** wing IVs: IV1 = mean(front put, front call) and IV2 = mean(back put, back call). Liquidity is the thinnest of all four legs.

In [ ]:
def norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def delta(side, spot, strike, iv, t, rate=0.0):
    """Option delta; with no time or vol left it is the intrinsic step (1 or 0 for a call)."""
    if iv <= 0 or t <= 0:
        call = 1.0 if spot > strike else 0.0
    else:
        d1 = (math.log(spot / strike) + (rate + iv**2 / 2.0) * t) / (iv * math.sqrt(t))
        call = norm_cdf(d1)
    return call if side == "call" else call - 1.0


def quote_at_delta(side_frame, side, spot, t, target, strikes, rate=0.0):
    """The row, among ``strikes``, whose delta is closest to ``target`` (e.g. 0.35 or -0.35)."""
    rows = side_frame[side_frame["strike"].isin(strikes)]
    deltas = pd.Series(
        [delta(side, spot, float(k), float(iv), t, rate) for k, iv in zip(rows["strike"], rows["iv"])],
        index=rows.index,
    ).dropna()
    if deltas.empty:
        return None
    return rows.loc[(deltas - target).abs().idxmin()]


def double_legs(front, back, spot, as_of, params=ScanParams()):
    """(front put, front call), (back put, back call) at the front expiry's -/+ delta strikes."""
    t1 = (front.expiry - as_of).days / params.calculator.days_per_year
    nears, fars = [], []
    for side, target in (("put", -params.double_delta), ("call", params.double_delta)):
        listed = listed_strikes((front, back), (side,))
        near = quote_at_delta(front.side(side), side, spot, t1, target, listed, params.rate)
        if near is None:
            return None
        nears.append(near)
        fars.append(nearest_quote(back.side(side), float(near["strike"])))
    return (nears[0], nears[1]), (fars[0], fars[1])


double = double_legs(front, back, spot, AS_OF, params)
if double is None:
    raise CalculatorError("No strike listed on both legs has a usable delta.")
wing_nears, wing_fars = double
if any(is_stale(q) for q in (*wing_nears, *wing_fars)):
    raise stale_error(f"the double calendar wings on {demo_front} / {demo_back}")

t1 = dte_of(demo_front) / params.calculator.days_per_year
pd.DataFrame([
    {"wing": side, "strike": float(n["strike"]), "front iv": float(n["iv"]),
     "front delta": delta(side, spot, float(n["strike"]), float(n["iv"]), t1, params.rate), "back iv": float(f["iv"])}
    for side, n, f in zip(("put", "call"), wing_nears, wing_fars)
])

In [ ]:
double_candidate = make_candidate(
    "double_35d", tuple(float(q["strike"]) for q in wing_nears), False, wing_nears, wing_fars, front, back, AS_OF, earnings, params
)
print(f"35-delta double calendar @ {'/'.join(f'{k:,.2f}' for k in double_candidate.strikes)}: {demo_front} / {demo_back}")
print(format_breakdown(double_candidate.result))
print("flags:", ", ".join(double_candidate.flags) or "none", "->", "PASS" if double_candidate.passed else "FAIL")

## 11. Putting it together: `scan()`

Everything above, joined into one function with **structured input and output**:

- `ScanInputs` holds all the data the scan needs: symbol, date, spot, chains and earnings date. It has **no network access**.
- `scan(inputs, params) -> ScanResult` is **pure**: the same inputs always give the same result.
  For every front/back pair it scores the call and put calendars in the strike window, skipping stale quotes, then the double calendar.
  Given `params.strike`, only that exact strike is scanned and the double calendar is skipped.
- `run(symbol, provider)` is the convenience wrapper: fetch through a provider, then `scan`. It is what the terminal command calls.

**Ranking.** Trades are sorted by `(blocked, FF undefined, −FF)`:

1. Trades **not blocked** by earnings, volume or open interest come first. A low FF alone doesn't push a trade down, so a clean 0.18 ranks above a 0.50 across earnings.
2. Within each group, highest FF first, with undefined FF last.

This is the shape of `src/scanners/forward_factor.py` in chrysos-lab.

In [ ]:
@dataclass(frozen=True)
class ScanInputs:
    symbol: str
    as_of: date
    spot: float
    chains: list
    earnings: date | None = None  # next earnings date on or after as_of


@dataclass(frozen=True)
class ScanResult:
    symbol: str
    as_of: date
    spot: float
    front_window: tuple
    back_window: tuple
    widened: bool
    min_ff: float
    min_volume: float
    min_open_interest: float
    earnings: date | None
    strike: float | None  # the only strike scanned; None means a window around spot
    strike_range: int  # 0 when a strike was given
    candidates: list = field(default_factory=list)  # best first
    skipped_stale: int = 0  # trades dropped because a quote was stale


def rank(candidate):
    blocked = bool(set(candidate.flags) - {"ff"})
    ff = candidate.result.ff
    return (blocked, ff is None, 0.0 if ff is None else -ff.value)


def unlisted_strike(strike, shared):
    below = max((k for k in shared if k < strike), default=None)
    above = min((k for k in shared if k > strike), default=None)
    nearest = ", ".join(f"{k:,.2f}" for k in (below, above) if k is not None)
    return f"No expiry pair lists the {strike:,.2f} strike on both legs. Nearest strikes that do: {nearest}."


def scan(inputs, params=ScanParams()):
    legs = pick_legs((c.expiry for c in inputs.chains), inputs.as_of, params)
    chains = {c.expiry: c for c in inputs.chains}

    candidates, skipped = [], 0
    shared = set()  # strikes listed on both legs of some pair
    for front_expiry in legs.front:
        for back_expiry in legs.back:
            if back_expiry <= front_expiry:
                continue
            front, back = chains[front_expiry], chains[back_expiry]
            atm_strikes = strikes_around((front, back), inputs.spot, 0, sides=("call",))

            for side in SIDES:
                listed = listed_strikes((front, back), (side,))
                shared |= listed
                if params.strike is None:
                    window = strikes_around((front, back), inputs.spot, params.strike_range, sides=(side,))
                else:
                    window = [k for k in listed if math.isclose(k, params.strike)]
                for strike in window:
                    near = nearest_quote(front.side(side), strike)
                    far = nearest_quote(back.side(side), strike)
                    if is_stale(near) or is_stale(far):
                        skipped += 1
                        continue
                    atm = side == "call" and strike in atm_strikes
                    candidates.append(make_candidate(
                        side, (strike,), atm, (near,), (far,), front, back, inputs.as_of, inputs.earnings, params
                    ))

            if params.strike is not None:
                continue
            double = double_legs(front, back, inputs.spot, inputs.as_of, params)
            if double is None:
                continue
            nears, fars = double
            if any(is_stale(q) for q in (*nears, *fars)):
                skipped += 1
                continue
            wing_strikes = tuple(float(q["strike"]) for q in nears)
            candidates.append(make_candidate(
                "double_35d", wing_strikes, False, nears, fars, front, back, inputs.as_of, inputs.earnings, params
            ))

    if not candidates:
        if skipped:
            raise stale_error(f"all {skipped} candidate trades")
        if params.strike is not None and shared:
            raise CalculatorError(unlisted_strike(params.strike, shared))
        raise CalculatorError("No strike is listed on both legs of any expiry pair.")

    return ScanResult(
        symbol=inputs.symbol, as_of=inputs.as_of, spot=inputs.spot,
        front_window=legs.front_window, back_window=legs.back_window, widened=legs.widened,
        min_ff=params.calculator.min_ff, min_volume=params.min_volume, min_open_interest=params.min_open_interest,
        earnings=inputs.earnings, strike=params.strike,
        strike_range=params.strike_range if params.strike is None else 0,
        candidates=sorted(candidates, key=rank), skipped_stale=skipped,
    )


def run(symbol, provider, params=ScanParams(), as_of=None):
    symbol = symbol.strip().upper()
    if not symbol:
        raise CalculatorError("No stock symbol provided.")
    as_of = as_of or market_date()
    expiries = provider.expirations(symbol)
    if not expiries:
        raise CalculatorError(f"No options found for stock symbol '{symbol}'.")
    legs = pick_legs(expiries, as_of, params)
    chains = [provider.option_chain(symbol, e) for e in sorted({*legs.front, *legs.back})]
    spot = provider.spot(symbol)
    if spot is None:
        raise CalculatorError("Unable to retrieve underlying stock price.")
    return scan(ScanInputs(symbol, as_of, spot, chains, provider.next_earnings(symbol, as_of)), params)

Run `scan` on the **same data** fetched above and confirm it matches the step-by-step values.

In [ ]:
result = scan(ScanInputs(symbol, AS_OF, spot, list(chains.values()), earnings), params)


def find(structure, strikes):
    return next(c for c in result.candidates if c.structure == structure and c.strikes == strikes
                and (c.front_expiry, c.back_expiry) == (demo_front, demo_back))


assert find("call", (atm_strike,)) == atm_candidate
assert find("double_35d", double_candidate.strikes) == double_candidate
passing = sum(c.passed for c in result.candidates)
print(f"scan() matches the step-by-step calculation: {len(result.candidates)} trades, {passing} pass, "
      f"{result.skipped_stale} skipped as stale.")

## 12. FF across every expiry pair

The ATM call calendar of every pair at a glance: rows are front expiries, columns are back expiries.
Blue cells clear the FF pass mark, red cells fall short, and grey sits right at it. Each cell also names the flags that fail it.

In [ ]:
atm_trades = [c for c in result.candidates if c.atm]
fronts = sorted({c.front_expiry for c in atm_trades})
backs = sorted({c.back_expiry for c in atm_trades})
grid = np.full((len(fronts), len(backs)), np.nan)
for c in atm_trades:
    if c.result.ff is not None:
        grid[fronts.index(c.front_expiry), backs.index(c.back_expiry)] = c.result.ff.value

min_ff = params.calculator.min_ff
spread = max(np.nanmax(np.abs(grid - min_ff)), 0.05) if np.isfinite(grid).any() else 0.05
ff_colors = LinearSegmentedColormap.from_list("ff", ["#e34948", "#f0efec", "#2a78d6"])  # below / at / above the pass mark
ff_colors.set_bad("#d9d8d4")  # undefined FF or no ATM trade

fig, ax = plt.subplots(figsize=(1.5 * len(backs) + 3, 0.8 * len(fronts) + 2))
image = ax.imshow(grid, cmap=ff_colors, norm=TwoSlopeNorm(vcenter=min_ff, vmin=min_ff - spread, vmax=min_ff + spread))
for c in atm_trades:
    ff = "n/a" if c.result.ff is None else f"{c.result.ff.value:.2f}"
    verdict = "PASS" if c.passed else ",".join(c.flags)
    ax.text(backs.index(c.back_expiry), fronts.index(c.front_expiry), f"{ff}\n{verdict}", ha="center", va="center", fontsize=8, color="#0b0b0b")
ax.set_xticks(range(len(backs)), [f"{e}\n{dte_of(e)}d" for e in backs])
ax.set_yticks(range(len(fronts)), [f"{e}  {dte_of(e)}d" for e in fronts])
ax.set_xlabel("Back expiry"); ax.set_ylabel("Front expiry")
fig.colorbar(image, ax=ax, label=f"FF (pass mark {min_ff:g})")
ax.set_title(f"{symbol} ATM call calendar FF by expiry pair ({DATA_SOURCE} data)")
plt.show()

## 13. Output table

This is the same layout as the chrysos-lab terminal command (`uv run forward-factor AAPL`): a summary per structure,
the ranked trades, then the full working for the top three.

In [ ]:
DETAIL = 3  # candidates given a full breakdown
LABELS = {"call": "call", "put": "put", "double_35d": "dbl 35d"}
NAMES = {"call": "call calendar", "put": "put calendar", "double_35d": "35-delta double calendar"}
TYPE_HEADERS = ("Type", "Trades", "Pass", "Best FF", "Best trade")
RANK_HEADERS = ("Rank", "Type", "Front", "DTE", "Back", "DTE", "Strike", "IV1", "IV2", "FF", "Vol", "OI", "Close by", "Result")


def strikes_text(c):
    return "/".join(f"{k:,.2f}" for k in c.strikes)


def trade_text(c):
    return f"{c.front_expiry} ({c.front_dte}d) / {c.back_expiry} ({c.back_dte}d) @ {strikes_text(c)}"


def describe(c):
    name = NAMES[c.structure] + (" (ATM)" if c.atm else "")
    return f"{name}  {trade_text(c)}  |  close by {c.close_by}"


def type_row(r, structure):
    """Trade count, passes and best trade for one structure (candidates are already ranked)."""
    trades = [c for c in r.candidates if c.structure == structure]
    best = trades[0] if trades else None
    return (
        LABELS[structure],
        str(len(trades)),
        str(sum(1 for c in trades if c.passed)),
        "n/a" if best is None or best.result.ff is None else f"{best.result.ff.value:.3f}",
        "" if best is None else trade_text(best),
    )


def rank_row(i, c):
    ff = c.result.ff
    return (
        str(i), "call ATM" if c.atm else LABELS[c.structure],
        str(c.front_expiry), str(c.front_dte), str(c.back_expiry), str(c.back_dte), strikes_text(c),
        f"{c.result.iv1:.2%}", f"{c.result.iv2:.2%}", "n/a" if ff is None else f"{ff.value:.3f}",
        f"{c.volume.value:,.0f}", f"{c.open_interest.value:,.0f}", str(c.close_by),
        "PASS" if c.passed else f"FAIL {','.join(c.flags)}",
    )


def strike_line(r):
    if r.strike is not None:
        return f"Strike: {r.strike:,.2f} only (pairs listing it on both legs)  |  double calendar skipped"
    plural = "" if r.strike_range == 1 else "s"
    return f"Strikes: nearest spot +/- {r.strike_range} step{plural}"


def format_candidate(c, r):
    """Forward vol working plus the liquidity and earnings checks for one trade."""
    earnings = ("FAIL" if c.crosses_earnings else "PASS", "none found" if r.earnings is None else str(r.earnings), f"> {c.back_expiry}")
    rows = [
        *breakdown_rows(c.result),
        check_row("min leg volume", c.volume, "{:,.0f}"),
        check_row("min leg open int", c.open_interest, "{:,.0f}"),
        ("earnings", *earnings),
    ]
    return "\n".join(format_table(CHECK_HEADERS, rows))


def format_scan(r, top=10):
    passing = sum(1 for c in r.candidates if c.passed)
    earnings = (
        f"Next earnings {r.earnings}: trades whose back expiry is on/after it fail." if r.earnings else "Next earnings: none found."
    )
    lines = [
        r.symbol,
        f"{passing} of {len(r.candidates)} trades pass: FF >= {r.min_ff:g}, no earnings before the back expiry, "
        f"every leg volume >= {r.min_volume:,.0f} and open interest >= {r.min_open_interest:,.0f}",
        "",
        f"Spot {r.spot:,.2f}  |  DTE windows: front {window_label(r.front_window)}, back {window_label(r.back_window)}  |  As of {r.as_of}",
        strike_line(r),
        earnings,
    ]
    if r.widened:
        lines.append("Windows widened: no listed expiries fit the default ones.")
    if r.skipped_stale:
        lines.append(f"{r.skipped_stale} trades skipped for stale quotes.")

    lines += ["", *format_table(TYPE_HEADERS, [type_row(r, s) for s in STRUCTURES])]
    ranked = r.candidates[:top]
    lines += ["", *format_table(RANK_HEADERS, [rank_row(i, c) for i, c in enumerate(ranked, 1)])]
    for i, c in enumerate(ranked[:DETAIL], 1):
        lines += ["", f"#{i}  {describe(c)}", format_candidate(c, r)]
    return "\n".join(lines)


print(format_scan(result))

## 14. Scan a single strike

`ScanParams(strike=K)` scans only that exact strike and ranks the expiry pairs that list it on both legs.
It is the terminal command's `--strike`, handy when you already like a level or hold a position there. There is no strike window and no double calendar.

The strike must match a listed one exactly. A strike nobody lists raises an error naming the nearest strikes that are.

In [ ]:
one_strike = scan(ScanInputs(symbol, AS_OF, spot, list(chains.values()), earnings), replace(params, strike=atm_strike))
print(format_scan(one_strike, top=5).split("\n\n#1")[0])

try:
    scan(ScanInputs(symbol, AS_OF, spot, list(chains.values()), earnings), replace(params, strike=atm_strike + 0.5))
except CalculatorError as e:
    print("\nError:", e)

## 15. Screen several tickers

Now that each piece is understood, `run()` does the whole thing in one call. Errors such as stale quotes or missing options
are caught per ticker, so one bad symbol doesn't stop the loop. `top` is the best-ranked trade, which may still fail.

In [ ]:
TICKERS = ["AAPL", "SPY", "TSLA", "AMD"] if DATA_SOURCE == "yahoo" else ["DEMO"]

rows = []
for t in TICKERS:
    try:
        r = run(t, provider, params, AS_OF)
        top = r.candidates[0]
        rows.append({"symbol": r.symbol, "trades": len(r.candidates), "pass": sum(c.passed for c in r.candidates),
                     "top trade": f"{LABELS[top.structure]} {trade_text(top)}",
                     "top FF": None if top.result.ff is None else round(top.result.ff.value, 3),
                     "top result": "PASS" if top.passed else "FAIL " + ",".join(top.flags),
                     "earnings": r.earnings})
    except CalculatorError as e:
        rows.append({"symbol": t, "top result": f"Error: {e}"})

pd.DataFrame(rows)

### Try it yourself

- Change `SYMBOL` at the top and use **Runtime → Run all**.
- Widen the strike window: `params = ScanParams(strike_range=2)` is the terminal command's `--range 2`.
- Demand more liquidity: `params = ScanParams(min_open_interest=5_000)` is `--min-oi 5000`.
- Raise the pass mark: `params = ScanParams(calculator=FFParams(min_ff=0.3))`.
- Look at shorter calendars: `params = ScanParams(front_window=(7, 21), back_window=(28, 45))`.
- Set `DATA_SOURCE = "synthetic"` to explore the steps when US markets are closed.